In [1]:
!pip install -U langchain langchain-core langchain-community langchain-google-genai google-genai python-dotenv
!pip install -U sentence-transformers
!pip install -U ultralytics opencv-python

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 44.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.6/57.6 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 47.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 36.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 2.9 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
  Attempting uninstall: google-ai-generativelanguage
    Found existing installation: google-ai-generativelanguage 0.6.15
    Uninstalling google-ai-generativelanguage-0.6.15:
      Successfully uninstalled google-ai-generativelanguage-0.6.15
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the 

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 35.8 MB/s eta 0:00:00


In [12]:
import os, io, base64
from pathlib import Path
from typing import List, Dict
from PIL import Image
from ultralytics import YOLO
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import HumanMessage
from dotenv import load_dotenv

load_dotenv()

llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0.4)
yolo_model = YOLO("yolo11n.pt")

In [13]:
# image -> base64 반환
def pil_to_data_urlFunc(img:Image.Image, format:str="jpeg") -> str:
  buf = io.BytesIO()
  img.save(buf, format=format)    # PIL 이미지를 지정 포멧으로 메모리 버퍼에 저장
  img_bytes = buf.getvalue()
  b64 = base64.b64encode(img_bytes).decode("utf-8")
  return f"data:image/jpeg;base64,{b64}"    # LLM이 이해할 수 있는 data URL 형태


# 음식 감지
def detect_dishesFunc(image_path:str, conf_thres:float=0.3) -> List[Dict]:
  results = yolo_model(image_path)[0]
  image = Image.open(image_path).convert("RGB")
  dishes = []
  boxes = results.boxes
  if boxes is None:
    return dishes

  names = results.names
  # print(f"names : {names}")

  for box, cls, conf in zip(boxes.xyxy, boxes.cls, boxes.conf):
    if conf < conf_thres:    # 신뢰도가 임계값 보다 낮으면
      continue

    x1, y1, x2, y2 = map(int, box.tolist())
    crop = image.crop((x1, y1, x2, y2))
    label = names[int(cls)]
    # print(label)
    dishes.append({
        "crop":crop,
        "label":label,
        "conf":float(conf),
        "bbox":[x1, y1, x2, y2]
    })
    # conf 별 내림차순
    dishes.sort(key=lambda d:d["conf"], reverse=True)
  return dishes

# llm에게 레시피 요청
def ask_recipe_with_llmFunc(dish_img:Image.Image, label:str | None = None) -> str:
  data_url = pil_to_data_urlFunc(dish_img)
  base_prompt = (
      "너는 전문 요리사이자 레시피 개발자야.\n"
      "다음 음식 사진을 보고 아래 내용을 한국어로 설명해줘.\n"
      "1. 이 요리의 이름(추정)과 특징을 요리 초보 중학생도 이해할 수 있개 설명\n"
      "2. 필요한 재료 목록\n"
      "3. 조리 순서를 단계별로 친절하게 설명\n"
      "4. 특별히 맛있게 만드는 킥도 알려줘\n"
      "5. 이 요리 조리법을 변형하여 할 수 있는 비슷한 요리 3가지 추천해줘\n"
      "중요사항 : 마크다운 문법(**, *, -) 등을 사용하지 말고 깔끔한 평문으로 답을 해줘"
  )
  if label:
    base_prompt += f"\n참고:YOLO 모델이 이 음식을 '{label}'로 감지함. 얘를 최대한 활용"
  msg = HumanMessage(
      content = [
          {"type":"text", "text":base_prompt},
          {"type":"image_url", "image_url":data_url}
      ]
  )
  resp = llm.invoke([msg])
  return resp.content


# 전체 파이프라인
def process_food_imageFunc(image_path:str, max_dishes:int=2) -> str:
  print(f"입력 이미지 : {image_path}")
  dishes = detect_dishesFunc(image_path)
  if not dishes:
    print("음식 이미지가 없어요. 전체 이미지를 그대로 LLM에게 전달합니다")
    whole_img = Image.open(image_path).convert("RGB")
    answer = ask_recipe_with_llmFunc(whole_img, label=None)
    print(f"\nLLM응답(전체이미지) =\n{answer}")
    return answer
  print(f"감지된 음식 수 : {len(dishes)}")

  # 감지된 리스트 중 앞에서부터 max_dishes개 만큼 잘라서 사용
  for i, dish in enumerate(dishes[:max_dishes], start=1):
    label=dish["label"]
    conf = dish["conf"]
    bbox = dish["bbox"]
    print(f"\n[{i}] 감지된 음식 --- ")
    print(f"라벨 {label}, 신뢰도: {conf:3f}, bbox: {bbox}")

    # 감지된 음식 이미지 레시피 추천 요청
    answer = ask_recipe_with_llmFunc(dish["crop"], label=None)
    print(f"\nLLM응답(라벨({label})이미지) =\n {answer}")

process_food_imageFunc("food.jpeg", max_dishes=1)


입력 이미지 : food.jpeg

image 1/1 /content/food.jpeg: 384x640 1 pizza, 151.6ms
Speed: 3.0ms preprocess, 151.6ms inference, 1.5ms postprocess per image at shape (1, 3, 384, 640)
감지된 음식 수 : 1

[1] 감지된 음식 --- 
라벨 pizza, 신뢰도: 0.946318, bbox: [46, 4, 1451, 832]

LLM응답(라벨(pizza)이미지) =
 안녕하세요! 여러분의 전문 요리사이자 레시피 개발자입니다. 사진 속 피자를 보니 정말 신선하고 먹음직스러워 보이네요! 이 피자를 맛있게 만들고 즐길 수 있도록 제가 자세히 설명해 드릴게요.

1.  이 요리의 이름(추정)과 특징을 요리 초보 중학생도 이해할 수 있게 설명

이 피자의 이름은 '신선한 토마토 부라타 피자'라고 부르면 딱 맞을 것 같아요. 우리가 흔히 먹는 피자는 빨간 토마토소스가 기본으로 깔려 있잖아요? 그런데 이 피자는 소스 대신 싱싱한 빨간 토마토를 얇게 썰어서 피자 위에 가득 올린 게 가장 큰 특징이에요. 마치 여름날 시원한 샐러드를 피자로 만든 것 같은 느낌이죠.

그리고 하얀 구름처럼 예쁘게 올라간 치즈는 '부라타 치즈'라고 하는데, 겉은 쫄깃하고 속은 부드러운 생크림 같은 느낌이라 입안에서 사르르 녹는 맛이 일품이에요! 초록색 바질 잎이 더해져서 색깔도 예쁘고, 향긋한 냄새까지 더해져서 정말 상큼하고 고급스러운 맛을 즐길 수 있답니다. 일반 피자보다 훨씬 가볍고 신선한 느낌이라, 특히 따뜻한 날씨에 먹으면 더 맛있을 거예요.

2.  필요한 재료 목록 (1판 기준)

*   피자 도우 1개 (시판용 냉동 도우나 생 도우를 사용하면 편리해요)
*   신선한 토마토 2~3개 (단단하고 잘 익은 토마토가 좋아요, 얇게 슬라이스해주세요)
*   부라타 치즈 1~2개 (모짜렐라 치즈로 대체 가능하지만, 부라타가 더 부드럽고 맛있어요)
*   신선한 바질 잎 